In [9]:
import os
import re
import sys
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)


warnings.filterwarnings("ignore")
CSV_PATH = os.environ.get("LOAN_CSV", "dataset/SingleCycleLoaneeMember.csv")


In [10]:
# from basemodel_notebook

LADDER = np.array([5, 10, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 120, 150,
                   200, 250, 300, 400, 500, 800, 1000, 2000, 5000]) * 1000

FEATURE_SETS = {
    "1 income": ["monthly_income"],
    "2 +purpose": ["monthly_income", "purpose"],
    "3 +land": ["monthly_income", "purpose", "land"],
    "4 +family size": ["monthly_income", "purpose", "land", "family_size"],
    "5 +earners,expenses": ["monthly_income", "purpose", "land", "family_size",
                            "earners", "yearly_expenses"],
    "6 +district": ["monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district"],
    "7 +gender": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", 'gender'
    ]
}
DEFAULT_SET = "4 +family size"
CAT_FEATURES = ["purpose", "district", 'gender']

def clean_purpose(s: pd.Series) -> pd.Series:
    """'Dairy Farm (ডেইরি ফার্ম)' -> 'Dairy Farm'"""
    return s.fillna("NA").astype(str).str.replace(r"\s*\(.*$", "", regex=True).str.strip()


def fit_meta(train_df: pd.DataFrame) -> dict:
    """Learn the cleaning constants from TRAINING data only."""
    purpose = clean_purpose(train_df["loan_use_sources"])
    counts = purpose.value_counts()
    district = train_df["current_addr_district"].astype(str).str.strip()
    gender =  train_df['gender'].astype(str).str.strip()
    return {
        "income_cap": float(train_df["total_monthly_income"].quantile(0.999)),
        "expense_cap": float(train_df["yearly_total_expenses"].quantile(0.999)),
        "land_sentinel": 1e5,  # values above this are entry errors (e.g. 2,147,483,647)
        "purposes": sorted(counts[counts >= 100].index.tolist()),
        "districts": sorted(district.unique().tolist()),
        "gender" : sorted(gender.unique().tolist())
    }

def build_features(df: pd.DataFrame, meta: dict, features=None) -> pd.DataFrame:
    """Build only the requested features, so unused input columns are not required."""
    builders = {
        "monthly_income": lambda: df["total_monthly_income"].clip(upper=meta["income_cap"]),
        "purpose": lambda: pd.Categorical(
            (lambda p: p.where(p.isin(meta["purposes"]), "rare"))(clean_purpose(df["loan_use_sources"])),
            categories=meta["purposes"] + ["rare"]),                    # unseen/rare -> 'rare'
        "land": lambda: df["farm_land_amt"].where(df["farm_land_amt"] < meta["land_sentinel"]),
        "family_size": lambda: df["total_family_member"].clip(upper=12),
        "earners": lambda: df["total_earning_member"],
        "yearly_expenses": lambda: df["yearly_total_expenses"].clip(upper=meta["expense_cap"]),
        "district": lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["districts"])))(df["current_addr_district"].astype(str).str.strip()),
            categories=meta["districts"]),  # unseen -> missing
        'gender': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["gender"])))(df['gender'].astype(str))
        )
    }
    X = pd.DataFrame(index=df.index)
    for f in (features or list(builders)):
        X[f] = builders[f]()
    return X

def rung(v):
    return np.abs(np.log(np.asarray(v, float))[:, None] - np.log(LADDER)[None, :]).argmin(1)


def metrics(actual, pred) -> dict:
    a = np.asarray(actual, float)
    p = np.clip(np.asarray(pred, float), 4000, 5e6)
    la, lp = np.log1p(a), np.log1p(p)
    big = a >= 300000
    return {
        "MAE": float(np.mean(np.abs(p - a))),
        "MedianAE": float(np.median(np.abs(p - a))),
        "RMSLE": float(np.sqrt(np.mean((lp - la) ** 2))),
        "R2_log": float(1 - np.sum((la - lp) ** 2) / np.sum((la - la.mean()) ** 2)),
        "within_1_ladder_step": float(np.mean(np.abs(rung(p) - rung(a)) <= 1)),
        "exact_ladder_step": float(np.mean(rung(p) == rung(a))),
        "AUC_loan_ge_300k": float(roc_auc_score(big, np.log(p))) if 0 < big.sum() < len(a) else float("nan"),
    }

In [11]:
# Same time-ordered 80/20 split as the base model
df = pd.read_csv(CSV_PATH)
df["_adm"] = pd.to_datetime(df["admission_date"])
df = df.sort_values(["_adm", "member_id"]).reset_index(drop=True)
y = df["loan_amount"].to_numpy(float)
cut = int(len(df) * 0.8)
tr, te = df.iloc[:cut], df.iloc[cut:]

meta = fit_meta(tr)                                   # constants learned from TRAIN only
Xtr, Xte = build_features(tr, meta), build_features(te, meta)
F4 = FEATURE_SETS["4 +family size"]                   # the base model's features
F6 = FEATURE_SETS["6 +district"]                      # + earners, expenses, district

In [13]:

def make_regressor(seed=0, **kw):
    """Same settings as the base model, but learning_rate 0.1 so experiments run faster."""
    p = dict(objective="l1", learning_rate=0.1, num_leaves=15, min_child_samples=100,
             subsample=0.8, subsample_freq=1, reg_lambda=5, cat_smooth=30,
             min_data_per_group=50, cat_l2=10, n_estimators=3000, verbose=-1, random_state=seed)
    p.update(kw)
    return lgb.LGBMRegressor(**p)

def fit_reg(X, y_log, seed=0, **kw):
    """Fit on the first 85% of rows, early-stop on the most recent 15%."""
    k = int(len(X) * 0.85)
    m = make_regressor(seed, **kw)
    m.fit(X.iloc[:k], y_log[:k], eval_set=[(X.iloc[k:], y_log[k:])], eval_metric="l1",
          callbacks=[lgb.early_stopping(60, verbose=False)])
    return m

def short(r: dict) -> dict:
    keep = ("MAE", "RMSLE", "R2_log", "within_1_ladder_step", "exact_ladder_step", "AUC_loan_ge_300k")
    return {k: round(float(v), 3) for k, v in r.items() if k in keep}

### Experiment-  1

In [14]:
rows = []
for name, feats in [("4 features", F4), ("6 features (+earners,expenses,district)", F6)]:
    for frac, n_seeds in [(0.05, 3), (0.10, 3), (0.25, 2), (0.50, 1), (1.00, 1)]:
        n = int(cut * frac)
        results = []
        for s in range(n_seeds):
            # random subset, kept in time order so early stopping still uses the newest rows
            idx = (np.sort(np.random.RandomState(100 + s).choice(cut, n, replace=False))
                   if frac < 1 else np.arange(cut))
            m = fit_reg(Xtr.iloc[idx][feats], np.log1p(y[idx]), seed=s)
            results.append(metrics(y[cut:], np.expm1(m.predict(Xte[feats]))))
        r = pd.DataFrame(results)
        rows.append([name, n, r.MAE.mean(), r.MAE.std() if n_seeds > 1 else np.nan,
                     r.R2_log.mean(), r.within_1_ladder_step.mean()])
        print(rows[-1], flush=True)

table = pd.DataFrame(rows, columns=["features", "train_rows", "MAE", "MAE_sd", "R2_log", "within1"])
print("\n", table.round(3).to_string(index=False))
table.to_csv("learning_curve.csv", index=False)

['4 features', 4959, np.float64(43915.94140093796), np.float64(177.3143672089897), np.float64(0.47584378147496514), np.float64(0.4538892697286181)]
['4 features', 9919, np.float64(43143.822380131125), np.float64(162.9878744123365), np.float64(0.4906468428453728), np.float64(0.46140301356237484)]
['4 features', 24798, np.float64(41922.933623232064), np.float64(1.525440241558541), np.float64(0.5114501567579129), np.float64(0.4667728537441026)]
['4 features', 49597, np.float64(41361.19608409052), nan, np.float64(0.5230809845825417), np.float64(0.46921246824468726)]
['4 features', 99194, np.float64(41079.6653004388), nan, np.float64(0.5272003870168556), np.float64(0.4705028428565668)]
['6 features (+earners,expenses,district)', 4959, np.float64(42905.95715013687), np.float64(559.3804249289309), np.float64(0.5121750837173763), np.float64(0.46556984824656905)]
['6 features (+earners,expenses,district)', 9919, np.float64(41483.49893002854), np.float64(378.3617714274172), np.float64(0.53787582

### Experiment- 2

In [15]:
# (a) how many rows per amount?
counts = pd.Series(y[:cut]).value_counts()
print("distinct loan amounts (all rows):", len(np.unique(y)), "| in train:", len(counts))
for t in [5000, 1000, 300, 100, 30, 10]:
    print(f"amounts with >= {t:>4} rows in train: {(counts >= t).sum():>3}"
          f"  (cover {counts[counts >= t].sum() / cut:.1%} of loans)")

# (b) multiclass over the ladder
classes = np.sort(counts[counts >= 300].index.values)
K = len(classes)
print("classes used:", K, classes.astype(int).tolist())
nearest = lambda v: classes[np.abs(np.log(v)[:, None] - np.log(classes)[None, :]).argmin(1)]
y_cls = np.searchsorted(classes, nearest(y[:cut]))           # class index of each training loan

k = int(cut * 0.85)
clf = lgb.LGBMClassifier(objective="multiclass", num_class=K, learning_rate=0.1, num_leaves=15,
                         min_child_samples=100, subsample=0.8, subsample_freq=1, reg_lambda=5,
                         cat_smooth=30, min_data_per_group=50, cat_l2=10, n_estimators=800,
                         verbose=-1, random_state=0)
clf.fit(Xtr.iloc[:k][F4], y_cls[:k], eval_set=[(Xtr.iloc[k:][F4], y_cls[k:])],
        callbacks=[lgb.early_stopping(30, verbose=False)])
P = clf.predict_proba(Xte[F4])                               # probability of each amount class

# four ways to turn a probability distribution into one number
arg = classes[P.argmax(1)]                                   # most likely amount
median = classes[(P.cumsum(1) >= 0.5).argmax(1)]             # median of the distribution
geo = np.exp((P * np.log(classes)[None, :]).sum(1))          # geometric mean
mean = (P * classes[None, :]).sum(1)                         # expected value

reg = fit_reg(Xtr[F4], np.log1p(y[:cut]))
reg_pred = np.expm1(reg.predict(Xte[F4]))
a = y[cut:]
R = {"classifier: argmax (most likely amount)": metrics(a, arg),
     "classifier: median of predicted distribution": metrics(a, median),
     "classifier: geometric mean": metrics(a, geo),
     "classifier: expected value": metrics(a, mean),
     "regression (base model)": metrics(a, reg_pred)}
print("\n", pd.DataFrame({k_: short(v) for k_, v in R.items()}).T.round(3).to_string())

big = a >= 300000
for nm, p in [("argmax", arg), ("median-of-dist", median), ("regression", reg_pred)]:
    print(f"{nm:15s} MAE normal {np.mean(np.abs(p - a)[~big]):8.0f} | MAE big {np.mean(np.abs(p - a)[big]):8.0f}"
          f" | median pred for big loans {np.median(p[big]):8.0f} | for <=30k loans {np.median(p[a <= 30000]):8.0f}")

distinct loan amounts (all rows): 120 | in train: 116
amounts with >= 5000 rows in train:   8  (cover 78.1% of loans)
amounts with >= 1000 rows in train:  13  (cover 94.0% of loans)
amounts with >=  300 rows in train:  21  (cover 97.5% of loans)
amounts with >=  100 rows in train:  27  (cover 98.5% of loans)
amounts with >=   30 rows in train:  44  (cover 99.5% of loans)
amounts with >=   10 rows in train:  61  (cover 99.8% of loans)
classes used: 21 [10000, 20000, 25000, 30000, 40000, 45000, 48000, 50000, 60000, 70000, 80000, 90000, 95000, 100000, 120000, 150000, 200000, 300000, 400000, 500000, 800000]

                                                     MAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
classifier: argmax (most likely amount)       44557.442  0.570   0.393                 0.459              0.277             0.919
classifier: median of predicted distribution  40527.723  0.508   0.517                 0.461              0.235             0.9

### Experiment- 3

In [16]:
a = y[cut:]
big_te = a >= 300000
ly = np.log1p(y)


def component_group(d):
    return np.where(d.loan_Component.str.startswith("AGROSOR"), "AGROSOR",
                    np.where(d.loan_Component.isin(["JAGORON", "SUFOLON"]), d.loan_Component, "other"))


g_tr, g_te = component_group(tr), component_group(te)
R = {}

# A. honest baseline
m = fit_reg(Xtr[F4], ly[:cut])
base = np.expm1(m.predict(Xte[F4]))
R["A. single model, 4 features (honest baseline)"] = metrics(a, base)

# B. component as an extra feature (LEAKY)
cats = sorted(set(g_tr))
Xc_tr, Xc_te = Xtr[F4].copy(), Xte[F4].copy()
Xc_tr["comp"] = pd.Categorical(g_tr, categories=cats)
Xc_te["comp"] = pd.Categorical(g_te, categories=cats)
mc = fit_reg(Xc_tr, ly[:cut])
R["B. single model + component as feature (LEAKY)"] = metrics(a, np.expm1(mc.predict(Xc_te)))

# C. one model per component, component known at prediction time (LEAKY)
pred_c = np.zeros(len(te))
for g in cats:
    i, j = np.where(g_tr == g)[0], np.where(g_te == g)[0]
    mg = fit_reg(Xtr.iloc[i][F4], ly[i], min_child_samples=min(100, max(20, len(i) // 50)))
    pred_c[j] = np.expm1(mg.predict(Xte.iloc[j][F4]))
    print(g, "train rows", len(i), "test rows", len(j), flush=True)
R["C. separate model per component, component known (LEAKY)"] = metrics(a, pred_c)

# D. legitimate two-stage: P(big) from features, then route to a normal or big regressor
is_big = y[:cut] >= 300000
k = int(cut * 0.85)
clf = lgb.LGBMClassifier(objective="binary", learning_rate=0.1, num_leaves=15, min_child_samples=100,
                         subsample=0.8, subsample_freq=1, reg_lambda=5, cat_smooth=30,
                         min_data_per_group=50, cat_l2=10, n_estimators=800, verbose=-1, random_state=0)
clf.fit(Xtr.iloc[:k][F4], is_big[:k], eval_set=[(Xtr.iloc[k:][F4], is_big[k:])],
        callbacks=[lgb.early_stopping(30, verbose=False)])
p = clf.predict_proba(Xte[F4])[:, 1]
m_normal = fit_reg(Xtr[~is_big][F4], ly[:cut][~is_big])
m_big = fit_reg(Xtr[is_big][F4], ly[:cut][is_big], min_child_samples=40)
log_n, log_b = m_normal.predict(Xte[F4]), m_big.predict(Xte[F4])
R["D1. two-stage, hard route at P(big)>0.5"] = metrics(a, np.expm1(np.where(p > 0.5, log_b, log_n)))
R["D2. two-stage, hard route at P(big)>0.35"] = metrics(a, np.expm1(np.where(p > 0.35, log_b, log_n)))
R["D3. two-stage, soft blend (log scale)"] = metrics(a, np.expm1(p * log_b + (1 - p) * log_n))

print("\n", pd.DataFrame({k_: short(v) for k_, v in R.items()}).T.round(3).to_string())
for nm, pr in [("A single", base), ("C per-component (leaky)", pred_c),
               ("D1 two-stage", np.expm1(np.where(p > 0.5, log_b, log_n))),
               ("D3 soft", np.expm1(p * log_b + (1 - p) * log_n))]:
    print(f"{nm:26s} MAE normal {np.mean(np.abs(pr - a)[~big_te]):9.0f} | MAE big {np.mean(np.abs(pr - a)[big_te]):9.0f}"
          f" | median pred for big {np.median(pr[big_te]):9.0f}")
print("AUC of P(big):", round(roc_auc_score(big_te, p), 3),
      "| big loans with P>0.5:", round((p[big_te] > 0.5).mean(), 3),
      "| precision at P>0.5:", round(big_te[p > 0.5].mean(), 3),
      "| non-big flagged:", round((p[~big_te] > 0.5).mean(), 4))

AGROSOR train rows 8829 test rows 2229
JAGORON train rows 72291 test rows 17987
SUFOLON train rows 17082 test rows 4367
other train rows 992 test rows 216

                                                                 MAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
A. single model, 4 features (honest baseline)             41079.665  0.503   0.527                 0.471              0.206             0.941
B. single model + component as feature (LEAKY)            33298.144  0.453   0.617                 0.503              0.248             0.998
C. separate model per component, component known (LEAKY)  31825.626  0.446   0.629                 0.509              0.244             0.999
D1. two-stage, hard route at P(big)>0.5                   42032.104  0.517   0.500                 0.465              0.205             0.929
D2. two-stage, hard route at P(big)>0.35                  42071.487  0.519   0.498                 0.468              0.211          

### Experiment- 4

In [17]:


purpose = clean_purpose(df["loan_use_sources"]).str.lower()


def purpose_family(s: str) -> str:
    if re.search(r"immigrant|abroad", s):
        return "Migration"
    if re.search(r"dairy|goru|cattle fatt|cow|goat|poultry|duck|sheep|livestock|animal|motataja|rearing|pigeon|farm(ing)?$", s):
        return "Livestock/Crop/Transport"
    if re.search(r"fish|shrimp|pond", s):
        return "Trade/Housing/Fishery"
    if re.search(r"paddy|vegetable cult|potato|jute|onion|betel|corn|maize|cultiv|wheat|mustard|chili|garlic|banana|mango|tobacco|crop|nursery|agricultur|seed", s):
        return "Livestock/Crop/Transport"
    if re.search(r"house|home|building|repair|toilet|latrine|tube|solar|sanitation|water", s):
        return "Trade/Housing/Fishery"
    if re.search(r"auto|vehicle|rickshaw|van|transport|truck|cng|bus$|tempo|motor|bike|boat|driver|easy bike|battery", s):
        return "Livestock/Crop/Transport"
    if re.search(r"business|shop|store|trade|dokan|stall|tea|grocery|cloth|garment|hardware|restaurant|hotel|medicine|pharmacy|distributor|mobile|electric|jewel|crockery|sanitary|brick|saw|mill|tailor|cosmetic|furniture|fertilizer|feed|wholesale|retail|bakery|food|fast|sweet|salon|parlor|barber|gas|plastic|glass", s):
        return "Trade/Housing/Fishery"
    return "Other"


family = purpose.map(purpose_family).values
print(pd.Series(family).value_counts(normalize=True).round(3).to_dict())

ly = np.log1p(y)
a = y[cut:]
single = np.expm1(fit_reg(Xtr[F4], ly[:cut]).predict(Xte[F4]))

separate = np.zeros(len(te))
for g in np.unique(family):
    i, j = np.where(family[:cut] == g)[0], np.where(family[cut:] == g)[0]
    mg = fit_reg(Xtr.iloc[i][F4], ly[i], min_child_samples=min(100, max(20, len(i) // 50)))
    separate[j] = np.expm1(mg.predict(Xte.iloc[j][F4]))
    print(g, "train rows", len(i), "test rows", len(j), flush=True)

R = {"single model (4 features)": metrics(a, single),
     "separate model per purpose family": metrics(a, separate)}
print("\n", pd.DataFrame({k: short(v) for k, v in R.items()}).T.round(3).to_string())
for g in np.unique(family):
    j = np.where(family[cut:] == g)[0]
    print(f"{g:26s} n={len(j):5d}  MAE single {np.mean(np.abs(single - a)[j]):9.0f}"
          f"  separate {np.mean(np.abs(separate - a)[j]):9.0f}")

{'Livestock/Crop/Transport': 0.684, 'Trade/Housing/Fishery': 0.257, 'Migration': 0.044, 'Other': 0.015}
Livestock/Crop/Transport train rows 67619 test rows 17162
Migration train rows 4446 test rows 994
Other train rows 1508 test rows 362
Trade/Housing/Fishery train rows 25621 test rows 6281

                                          MAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
single model (4 features)          41079.665  0.503   0.527                 0.471              0.206             0.941
separate model per purpose family  41175.193  0.505   0.524                 0.472              0.211             0.940
Livestock/Crop/Transport   n=17162  MAE single     30929  separate     31089
Migration                  n=  994  MAE single    100445  separate    102122
Other                      n=  362  MAE single     58574  separate     58092
Trade/Housing/Fishery      n= 6281  MAE single     58413  separate     58115
